## test embedding changes (incl metadata)

In [1]:
from chromadb import PersistentClient

# Test V3 metadata structure
client = PersistentClient(path='../data/chroma_db_june11_clean')
collection = client.get_collection('qa_v3_enhanced')

# Get a few documents to see metadata structure
results = collection.get(limit=10, include=['metadatas', 'documents'])

print("=== V3 Metadata Analysis ===")
print(f"Documents retrieved: {len(results['metadatas'])}")
print(f"Metadata keys: {list(results['metadatas'][0].keys())}")
print(f"Sample metadata: {results['metadatas'][0]}")
print(f"Sample content length: {len(results['documents'][0])}")

=== V3 Metadata Analysis ===
Documents retrieved: 10
Metadata keys: ['qid', 'qs', 'sid', 'typ']
Sample metadata: {'qid': 4974, 'qs': 6, 'sid': 0, 'typ': 2}
Sample content length: 131


In [5]:
# Check actual V3 content structure
print("=== V3 Content Analysis ===")
for i in range(9):
    print(f"\n--- Document {i+1} ---")
    print(f"Metadata: {results['metadatas'][i]}")
    print(f"Content: '{results['documents'][i]}'")
    print(f"Content length: {len(results['documents'][i])}")

=== V3 Content Analysis ===

--- Document 1 ---
Metadata: {'qid': 4974, 'qs': 6, 'sid': 0, 'typ': 2}
Content: 'Topic: General;;Ep 615;;Banks, Bikies and Broadband;;2009-04-09;;Diane Dent;;Do it now? I have noticed restaurants advertising this'
Content length: 131

--- Document 2 ---
Metadata: {'qs': 6, 'qid': 4974, 'typ': 1, 'sid': 0}
Content: 'Topic: General;;Ep 615;;Banks, Bikies and Broadband;;2009-04-09;;Tony Jones;;Why can't banks offer full interest rate cuts when the rates are cut by the Reserve? Jane Caro, put on your advertising hat and try and explain how a bank would actually get away with spinning this decision that they've taken?'
Content length: 303

--- Document 3 ---
Metadata: {'qs': 8, 'sid': 2890, 'typ': 3, 'qid': 4974}
Content: 'Topic: General;;Ep 615;;Banks, Bikies and Broadband;;2009-04-09;;Jane Caro;;Yeah, it's a bit of a PR disaster for them, isn't it? That's what I think. I just don't see how they're going to pull this one off at all. I mean, I understand why th

In [3]:
# Comprehensive topic analysis
def analyze_v3_topics(collection, sample_size=1000):
    """Analyze topic field population in V3 documents"""
    results = collection.get(limit=sample_size, include=['documents', 'metadatas'])
    
    topic_stats = {
        'total_docs': len(results['documents']),
        'empty_topics': 0,
        'populated_topics': 0,
        'topic_examples': [],
        'empty_examples': []
    }
    
    for i, content in enumerate(results['documents']):
        if content.startswith('Topic: ') and ';;' in content:
            # Extract topic field
            topic_part = content.split(';;')[0]
            topic_value = topic_part[7:] if topic_part.startswith('Topic: ') else ''
            
            if not topic_value.strip():
                topic_stats['empty_topics'] += 1
                if len(topic_stats['empty_examples']) < 5:
                    topic_stats['empty_examples'].append({
                        'doc_index': i,
                        'content_preview': content[:100] + '...'
                    })
            else:
                topic_stats['populated_topics'] += 1
                if len(topic_stats['topic_examples']) < 10:
                    topic_stats['topic_examples'].append({
                        'topic': topic_value,
                        'content_preview': content[:100] + '...'
                    })
    
    return topic_stats

# Run the analysis
stats = analyze_v3_topics(collection, 1000)
print(f"=== Topic Population Analysis ===")
print(f"Total documents analyzed: {stats['total_docs']}")
print(f"Empty topics: {stats['empty_topics']} ({stats['empty_topics']/stats['total_docs']*100:.1f}%)")
print(f"Populated topics: {stats['populated_topics']} ({stats['populated_topics']/stats['total_docs']*100:.1f}%)")

print(f"\n=== Topic Examples ===")
for example in stats['topic_examples']:
    print(f"Topic: '{example['topic']}' | Content: {example['content_preview']}")

print(f"\n=== Empty Topic Examples ===")
for example in stats['empty_examples']:
    print(f"Doc {example['doc_index']}: {example['content_preview']}")

=== Topic Population Analysis ===
Total documents analyzed: 1000
Empty topics: 0 (0.0%)
Populated topics: 1000 (100.0%)

=== Topic Examples ===
Topic: 'General' | Content: Topic: General;;Ep 615;;Banks, Bikies and Broadband;;2009-04-09;;Diane Dent;;Do it now? I have notic...
Topic: 'General' | Content: Topic: General;;Ep 615;;Banks, Bikies and Broadband;;2009-04-09;;Tony Jones;;Why can't banks offer f...
Topic: 'General' | Content: Topic: General;;Ep 615;;Banks, Bikies and Broadband;;2009-04-09;;Jane Caro;;Yeah, it's a bit of a PR...
Topic: 'General' | Content: Topic: General;;Ep 615;;Banks, Bikies and Broadband;;2009-04-09;;Tony Jones;;Let's hear from Andrew ...
Topic: 'General' | Content: Topic: General;;Ep 615;;Banks, Bikies and Broadband;;2009-04-09;;Andrew Boe;;Ivan, at least, needed ...
Topic: 'General' | Content: Topic: General;;Ep 615;;Banks, Bikies and Broadband;;2009-04-09;;John Hewson;;Well, look, I think th...
Topic: 'General' | Content: Topic: General;;Ep 615;;Banks, Bikie